In [39]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from collections import Counter

In [40]:
dataset_path = r"D:\Develop\dataset\CUBIT-Det"
split=["train","valid","test"]
num_classes=3

In [41]:
def count_split(split):
    lbl_dir=os.path.join(dataset_path,split,"labels")
    counts=Counter()
    total_files=0
    for fn in os.listdir(lbl_dir):
        total_files+=1
        path=os.path.join(lbl_dir,fn)
        with open(path) as f:
            for line in f:
                line=line.strip()
                if line:
                    parts=line.split()
                    cid=int(parts[0])
                    counts[cid]+=1
    return counts, total_files

In [42]:
def print_summary(all_counts):
    for split,(cnt,total_files) in all_counts.items():
        total_objs=sum(cnt.values())
        print(f"{split}:")
        print(f"label files: {total_files}, total objects: {total_objs}")
        for cid in range(num_classes):
            c=cnt.get(cid,0)
            pct=(c/total_objs)*100 if total_objs>0 else 0
            print(f"class {cid}: {c} objects, {pct:.2f}%")

In [43]:
all_counts={}
for s in split:
    print(f"Processing {s} split...")
    cnt,files=count_split(s)
    all_counts[s]=(cnt,files)
print_summary(all_counts)

Processing train split...
Processing valid split...
Processing test split...
train:
label files: 304, total objects: 736
class 0: 0 objects, 0.00%
class 1: 567 objects, 77.04%
class 2: 169 objects, 22.96%
valid:
label files: 35, total objects: 76
class 0: 0 objects, 0.00%
class 1: 58 objects, 76.32%
class 2: 18 objects, 23.68%
test:
label files: 82, total objects: 176
class 0: 0 objects, 0.00%
class 1: 139 objects, 78.98%
class 2: 37 objects, 21.02%


In [44]:
def cls0_percentage(split):
    lbl_dir=os.path.join(dataset_path,split,"labels")
    total_files=0
    cls0_count=0
    for fn in os.listdir(lbl_dir):
        total_files+=1
        path=os.path.join(lbl_dir,fn)
        with open(path) as f:
            for line in f:
                line=line.strip()
                cid=int(line.split()[0])
                if cid==0:
                    cls0_count+=1
                    break
    pct=(cls0_count/total_files)*100 if total_files>0 else 0
    return cls0_count, total_files, pct

In [45]:
for s in split:
    cls0_count, total_files, pct=cls0_percentage(s)
    print(f"{s}: {cls0_count}/{total_files} files contain class 0 ({pct:.2f}%)")

train: 0/304 files contain class 0 (0.00%)
valid: 0/35 files contain class 0 (0.00%)
test: 0/82 files contain class 0 (0.00%)


# 删除含crack类数据

In [46]:
from pathlib import Path

In [47]:
DATA_ROOT=Path(dataset_path)
IMG_EXTS = [".JPG"]
dry_run = False  # True 先预览，False 真删

to_delete = []
for split in split:
    lbl_dir = DATA_ROOT / split / "labels"
    img_dir = DATA_ROOT / split / "images"
    for lbl in sorted(lbl_dir.glob("*.txt")):
        contains0 = False
        try:
            with open(lbl) as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    parts = line.split()
                    try:
                        cid = int(parts[0])
                    except:
                        continue
                    if cid == 0:
                        contains0 = True
                        break
        except Exception as e:
            print("read error", lbl, e)
            continue
        if contains0:
            stem = lbl.stem
            # 找到同名图片（可能有多种扩展）
            imgs = [img_dir / (stem + ext) for ext in IMG_EXTS]
            imgs_exist = [p for p in imgs if p.exists()]
            to_delete.append((split, lbl, imgs_exist))

# 报告并执行
total_lbl = len(to_delete)
total_imgs = sum(len(imgs) for _, _, imgs in to_delete)
print(
    f"Found {total_lbl} label files containing class 0, {total_imgs} image files matched."
)
for split, lbl, imgs in to_delete:
    print(
        f"{split}: {lbl} -> {', '.join(str(p.name) for p in imgs) or '(no image found)'}"
    )

if not dry_run:
    for _, lbl, imgs in to_delete:
        try:
            lbl.unlink()
        except Exception as e:
            print("failed delete label", lbl, e)
        for p in imgs:
            try:
                p.unlink()
            except Exception as e:
                print("failed delete image", p, e)
    print("Deletion finished.")
else:
    print("Dry run: no files deleted. Set dry_run=False to delete.")

Found 0 label files containing class 0, 0 image files matched.
Deletion finished.


# bbox转mask

In [50]:
from pathlib import Path
import numpy as np
import cv2
import torch
from tqdm import tqdm

# SAM2 官方接口
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

In [51]:
# 路径配置
DATA_ROOT = Path(r"D:\Develop\dataset\CUBIT-Det")
OUT_ROOT = Path(r"D:\Develop\dataset\CUBIT-Det-seg")
SPLITS = ["train", "valid", "test"]
IMG_EXTS = [".jpg"]

TARGET_CLASSES = {1, 2}
PAD = 0.02

ckpt = r"D:\HP\OneDrive\Desktop\学校\课程\专业课\大数据综合工程设计\crack-seg\算法\sam2_hiera_base_plus.pt"

# 这个配置名要和权重匹配（base-plus）
# 若你本地 sam2 版本命名不同，改成对应的 b+/base_plus 配置名
model_cfg = "sam2_hiera_b+.yaml"

device = "cuda" if torch.cuda.is_available() else "cpu"
sam2_model = build_sam2(model_cfg, ckpt, device=device)
predictor = SAM2ImagePredictor(sam2_model)

OUT_ROOT.mkdir(parents=True, exist_ok=True)
mask_root = OUT_ROOT / "masks"
mask_root.mkdir(parents=True, exist_ok=True)


def read_yolo_bbox_line(line):
    parts = line.strip().split()
    cid = int(parts[0])
    cx, cy, w, h = map(float, parts[1:5])
    return cid, cx, cy, w, h


def find_image(img_dir, stem):
    for ext in IMG_EXTS:
        p = img_dir / f"{stem}{ext}"
        if p.exists():
            return p
    return None


def mask_to_polygon(mask):
    # mask: HxW bool/0-1
    m = mask.astype(np.uint8) * 255
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    c = max(contours, key=cv2.contourArea)
    if cv2.contourArea(c) < 10:
        return None
    eps = 0.01 * cv2.arcLength(c, True)
    approx = cv2.approxPolyDP(c, eps, True)
    pts = approx.reshape(-1, 2)
    if len(pts) < 3:
        return None
    return pts


written_labels = 0
written_images = 0
written_masks = 0
kept_instances = 0
skipped = 0

for split_name in SPLITS:
    lbl_dir = DATA_ROOT / split_name / "labels"
    img_dir = DATA_ROOT / split_name / "images"
    out_lbl_dir = OUT_ROOT / split_name / "labels"
    out_img_dir = OUT_ROOT / split_name / "images"
    out_mask_dir = mask_root / split_name

    out_lbl_dir.mkdir(parents=True, exist_ok=True)
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_mask_dir.mkdir(parents=True, exist_ok=True)

    for lbl_path in tqdm(sorted(lbl_dir.glob("*.txt")), desc=f"SAM2->{split_name}"):
        stem = lbl_path.stem
        img_path = find_image(img_dir, stem)
        if img_path is None:
            skipped += 1
            continue

        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            skipped += 1
            continue

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        predictor.set_image(img_rgb)

        out_lines = []
        instance_idx = 0

        with open(lbl_path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue

                cid, cx, cy, bw, bh = read_yolo_bbox_line(line)
                if cid not in TARGET_CLASSES:
                    continue
                if cid==2:
                    cid=5

                x1 = max(0, (cx - bw / 2 - PAD) * w)
                y1 = max(0, (cy - bh / 2 - PAD) * h)
                x2 = min(w, (cx + bw / 2 + PAD) * w)
                y2 = min(h, (cy + bh / 2 + PAD) * h)
                box = np.array([x1, y1, x2, y2], dtype=np.float32)

                with torch.inference_mode():
                    masks, scores, _ = predictor.predict(
                        point_coords=None,
                        point_labels=None,
                        box=box[None, :],  # shape (1,4)
                        multimask_output=True,
                    )

                if masks is None or len(masks) == 0:
                    continue

                if scores is not None and len(scores) > 0:
                    best = int(np.argmax(scores))
                else:
                    areas = [int(np.sum(m.astype(np.uint8))) for m in masks]
                    best = int(np.argmax(areas)) if areas else 0

                mask = masks[best]
                pts = mask_to_polygon(mask)
                if pts is None:
                    continue

                # crack-seg 分割标签行：class_id x1 y1 x2 y2 ...
                norm = [(float(x) / w, float(y) / h) for x, y in pts]
                coord_str = " ".join(f"{v:.12f}" for xy in norm for v in xy)
                out_lines.append(f"{cid} {coord_str}\n")
                kept_instances += 1

                mask_name = f"{stem}_{instance_idx}_cls{cid}.png"
                cv2.imwrite(
                    str(out_mask_dir / mask_name), (mask.astype(np.uint8) * 255)
                )
                written_masks += 1
                instance_idx += 1

        if out_lines:
            out_img = out_img_dir / img_path.name
            if not out_img.exists():
                out_img.write_bytes(img_path.read_bytes())
                written_images += 1

            with open(out_lbl_dir / f"{stem}.txt", "w", encoding="utf-8") as wf:
                wf.writelines(out_lines)
            written_labels += 1

print(
    f"Finished. labels={written_labels}, images={written_images}, "
    f"masks={written_masks}, instances={kept_instances}, skipped={skipped}"
)

SAM2->test: 100%|██████████| 82/82 [00:48<00:00,  1.68it/s]

Finished. labels=420, images=420, masks=988, instances=988, skipped=0
